# Milestone 04c — Separability Analysis

Extends Milestone 4 with three analyses motivated by the Fisher ratio result:
separability peaks at depth 1 then collapses for feedforward architectures,
while CORnet-S holds up across its recurrent timesteps.

**Analysis 1 — RSA:** ECoG × CNN layer RDMs across sliding time windows  
**Analysis 2 — Geodesic Fisher:** Isomap vs Euclidean category separability  
**Analysis 3 — Scrambled check:** late-onset gamma specificity

Each analysis section can be re-run independently after Cell 5 (epoch cache).

**Prerequisites:** run `milestone_04b_rerun_extraction.ipynb` first to build CNN caches.

In [ ]:
import sys, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import Isomap
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=UserWarning)

_CANDIDATES = [
    Path('/Users/winstonluk/Documents/NEURON/FinalProject'),
    Path('/sessions/modest-sweet-noether/mnt/FinalProject'),
    Path.cwd().parent,
]
ROOT = next(p for p in _CANDIDATES if (p / 'src' / 'cnn_features.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))

from cnn_features import (
    CATEGORY_NAMES,
    _md5_12,
    _read_run_images_and_trials,
    find_brands_run_mats,
    load_feature_cache,
)
from aim_4_1_encoding_pipeline import Config, load_broadband_run, epoch_trials

RESULTS  = ROOT / 'results'
STIMULI  = ROOT / 'data' / 'ds004194' / 'stimuli'

CATEGORY_COLORS = {
    'bodies':    '#e6194b',
    'buildings': '#3cb44b',
    'faces':     '#4363d8',
    'objects':   '#f58231',
    'scenes':    '#911eb4',
    'scrambled': '#a9a9a9',
}
ALL_ARCHS = ['resnet50', 'alexnet', 'vgg16', 'convnext_tiny', 'densenet121', 'swin_t', 'cornet_s']

# Load stimulus bank and build hash → image_id lookup
with open(RESULTS / 'cache' / 'stimulus_bank.pkl', 'rb') as f:
    bank = pickle.load(f)
hash_to_id = dict(zip(bank.metadata['hash'], bank.metadata['image_id']))
print(f'ROOT: {ROOT}')
print(f'Bank: {len(bank)} images, {bank.metadata["category"].value_counts().to_dict()}')


In [8]:
def load_visual_electrodes(atlas_path=RESULTS / 'milestone_01_electrode_atlas.csv'):
    """Return {subject: [ch_name, ...]} for Wang-atlas visual electrodes.

    Excludes FEF and unlabelled electrodes; keeps V1-V4, LO, VO, and
    any other labelled visual area that is not a frontal field.
    """
    df = pd.read_csv(atlas_path)
    # Identify the column that holds the Wang ROI label
    label_col = next((c for c in df.columns if 'wang' in c.lower() or 'roi' in c.lower()), None)
    if label_col is None:
        # Fallback: use first non-subject, non-electrode column
        label_col = [c for c in df.columns if c not in ('subject','electrode','x','y','z')][0]
    print(f'Using atlas label column: {label_col!r}')
    print('Unique labels:', df[label_col].dropna().unique().tolist())

    # Keep any labelled visual area that is NOT FEF / unlabelled / empty
    non_visual = {'', 'nan', 'fef', 'none', 'unknown'}
    vis_mask = df[label_col].astype(str).str.lower().apply(
        lambda v: bool(v) and v not in non_visual
    )
    vis_df = df[vis_mask].copy()

    # Build per-subject channel lists; subject column may be 'subject' or 'sub'
    sub_col = 'subject' if 'subject' in df.columns else 'sub'
    ch_col  = 'electrode' if 'electrode' in df.columns else 'channel'
    result = {}
    for sub in vis_df[sub_col].unique():
        chs = vis_df[vis_df[sub_col] == sub][ch_col].tolist()
        result[sub] = chs
        print(f'  {sub}: {len(chs)} visual electrodes')
    return result

vis_electrodes = load_visual_electrodes()


Using atlas label column: 'wang_label_max'
Unique labels: ['unassigned', 'PHC2', 'PHC1', 'IPS0', 'V3d', 'V3B', 'TO1', 'hV4', 'V2v', 'V1v', 'V2d', 'LO1', 'LO2', 'IPS3', 'IPS2', 'V1d', 'V3A', 'IPS1', 'FEF', 'V3v', 'VO1']
  p02: 56 visual electrodes
  p06: 142 visual electrodes
  p07: 120 visual electrodes
  p10: 252 visual electrodes
  p13: 116 visual electrodes
  p14: 92 visual electrodes


In [9]:
def build_trial_image_ids(mat_path):
    """Map each trial in a run's .mat file to an image_id (0-287).

    Returns (n_trials,) int array; -1 where hash lookup fails.
    """
    images, trialindex, _ = _read_run_images_and_trials(mat_path)
    ids = []
    for ti in trialindex:
        img = images[int(ti) - 1]          # 1-indexed → 0-indexed
        h = _md5_12(img.tobytes())
        ids.append(hash_to_id.get(h, -1))
    return np.array(ids, dtype=int)


In [ ]:
from aim_4_1_encoding_pipeline import load_events

def epoch_run(mat_path, sub, task, run, vis_chs, cfg):
    """Load one run, epoch around stimulus onsets, return (epochs, image_ids, t).

    epochs    : (n_valid_trials, n_vis_ch, n_t) float32
    image_ids : (n_valid_trials,) int  — image_id for each trial
    t         : (n_t,) float          — time in seconds relative to onset
    """
    # Broadband data (MNE BrainVision reader; correct session path)
    bb, info = load_broadband_run(cfg, sub, run, task)   # (n_ch, n_samp)

    # Select visual channels present in this recording
    all_chs = info['ch_names']
    picks_idx = [all_chs.index(c) for c in vis_chs if c in all_chs]
    if not picks_idx:
        return None, None, None
    bb_vis = bb[picks_idx]          # (n_vis_ch, n_samp)

    # Load BIDS events.tsv via pipeline (correct session path)
    try:
        events_df = load_events(cfg, sub, run, task)
    except FileNotFoundError as e:
        print(f'  [skip] events.tsv not found: {e}')
        return None, None, None

    # Build a info dict with only the visual channels
    vis_info = {'ch_names': [all_chs[i] for i in picks_idx], 'fs': info['fs']}

    epochs, ev_kept = epoch_trials(bb_vis, vis_info, events_df, cfg)
    # epochs: (n_valid, n_vis_ch, n_t)

    # Build time vector
    fs  = info['fs']
    n_t = epochs.shape[2]
    pre = int(round(-cfg.trial_window[0] * fs))
    t   = (np.arange(n_t) - pre) / fs

    # Map trials to image_ids via hash lookup
    trial_ids = build_trial_image_ids(mat_path)
    valid_row_idx = ev_kept.index.tolist()
    if len(valid_row_idx) != epochs.shape[0]:
        valid_row_idx = list(range(epochs.shape[0]))

    aligned_ids = np.array(
        [trial_ids[ri] if ri < len(trial_ids) else -1 for ri in valid_row_idx],
        dtype=int,
    )
    return epochs.astype(np.float32), aligned_ids, t


In [11]:
cfg = Config(bids_root=ROOT / 'data' / 'ds004194')

def load_subject_epochs(sub_label, cfg=cfg):
    """Build (288, n_vis_ch, n_t) trial-averaged epoch tensor for one subject.

    sub_label : e.g. 'p13' (without 'sub-' prefix)
    """
    cache_path = RESULTS / 'cache' / f'milestone_04c_epochs_{sub_label}.pkl'
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            d = pickle.load(f)
        print(f'Loaded {sub_label} from cache: {d["avg_epochs"].shape}')
        return d['avg_epochs'], d['t'], d['ch_names']

    sub = f'sub-{sub_label}'
    vis_chs = vis_electrodes.get(sub, vis_electrodes.get(sub_label, []))
    if not vis_chs:
        raise ValueError(f'No visual electrodes found for {sub_label}. '
                         f'Keys in atlas: {list(vis_electrodes.keys())}')

    mat_runs = find_brands_run_mats(STIMULI, subjects=[sub])
    if not mat_runs:
        raise FileNotFoundError(f'No stimulus .mat files found for {sub}')

    # Accumulate per image_id
    accumulator = {img_id: [] for img_id in range(288)}
    t_out = None
    n_ch  = len(vis_chs)

    for mat_path in mat_runs:
        # Parse run number and task from filename
        stem  = mat_path.stem
        parts = stem.split('_')
        task  = next(p.split('-', 1)[1] for p in parts if p.startswith('task-'))
        run   = int(next(p.split('-', 1)[1] for p in parts if p.startswith('run-')))

        epochs, image_ids, t = epoch_run(mat_path, sub, task, run, vis_chs, cfg)
        if epochs is None:
            continue
        if t_out is None:
            t_out = t

        for i, img_id in enumerate(image_ids):
            if img_id < 0:
                continue
            accumulator[img_id].append(epochs[i])    # (n_vis_ch, n_t)

        n_good = int((image_ids >= 0).sum())
        print(f'  {sub} task-{task} run-{run:02d}: {n_good} valid trials')

    # Average per image_id
    n_t = len(t_out) if t_out is not None else 665
    avg = np.full((288, n_ch, n_t), np.nan, dtype=np.float32)
    rep_counts = np.zeros(288, dtype=int)
    for img_id, trials in accumulator.items():
        if not trials:
            continue
        stack = np.stack(trials, axis=0)             # (n_reps, n_ch, n_t)
        avg[img_id] = np.nanmean(stack, axis=0)
        rep_counts[img_id] = len(trials)

    zero_reps = np.where(rep_counts == 0)[0]
    if len(zero_reps):
        print(f'  WARNING: {len(zero_reps)} images have 0 trials in {sub_label}')
    else:
        print(f'  {sub_label}: {rep_counts.min()}–{rep_counts.max()} reps/image, '
              f'{n_ch} visual channels, shape {avg.shape}')

    d = dict(avg_epochs=avg, t=t_out, ch_names=vis_chs)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'wb') as f:
        pickle.dump(d, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'  Cached → {cache_path}')
    return avg, t_out, vis_chs

print('Loading p13 epochs...')
avg_p13, t_epoch, chs_p13 = load_subject_epochs('p13')
print('Loading p14 epochs...')
avg_p14, _,       chs_p14 = load_subject_epochs('p14')

t_ms = t_epoch * 1000.0
print(f'\nTime axis: [{t_ms[0]:.0f}, {t_ms[-1]:.0f}] ms  ({len(t_ms)} samples)')


Loading p13 epochs...
Loaded p13 from cache: (288, 116, 665)
Loading p14 epochs...
Loaded p14 from cache: (288, 92, 665)

Time axis: [-100, 1197] ms  (665 samples)


## Analysis 1 — RSA: ECoG × CNN RDMs Across Time

Per-subject ECoG RDMs are computed independently at each 25ms sliding window, then
averaged before correlating with CNN layer RDMs. This avoids the SNR confound from
concatenating p13 (8 runs, ~8 reps/image) and p14 (5 runs, ~5 reps/image).

Inter-subject RDM reliability at each window is plotted below the heatmap as a
reliability ceiling — windows with near-zero inter-subject reliability indicate
noise-dominated ECoG geometry that should not be interpreted.

In [12]:
cnn_layer_order = []    # list of (arch, layer_key)
cnn_rdms        = {}    # (arch, layer_key) -> upper-triangle vector (41328,)

for arch in ALL_ARCHS:
    p = RESULTS / f'cnn_features_{arch}.pkl'
    if not p.exists():
        print(f'[skip] {arch} — cache not found')
        continue
    cache = load_feature_cache(p)
    for layer_key in cache.features.keys():
        X = StandardScaler().fit_transform(cache.features[layer_key].astype('float32'))
        cnn_rdms[(arch, layer_key)] = pdist(X, metric='correlation')
        cnn_layer_order.append((arch, layer_key))

n_cnn = len(cnn_layer_order)
print(f'CNN RDMs precomputed: {n_cnn} layers across {len(ALL_ARCHS)} archs')
print(f'Upper-triangle size: {len(next(iter(cnn_rdms.values())))} image pairs')


CNN RDMs precomputed: 42 layers across 7 archs
Upper-triangle size: 41328 image pairs


In [ ]:
# Sliding window parameters
WIN_MS, STEP_MS = 25, 25
T_START_MS, T_END_MS = -100, 1000

windows = np.array([[s, s + WIN_MS]
                    for s in range(T_START_MS, T_END_MS, STEP_MS)
                    if s + WIN_MS <= T_END_MS])
win_centers_ms = windows.mean(axis=1)
n_win = len(windows)
print(f'{n_win} windows × {n_cnn} CNN layers')

rsa_matrix    = np.full((n_win, n_cnn), np.nan, dtype=np.float32)
reliability   = np.full(n_win, np.nan, dtype=np.float32)

def _window_resp(avg_epochs, t_ms, ws, we):
    """Mean response in window [ws, we) ms. Returns (288, n_ch)."""
    mask = (t_ms >= ws) & (t_ms < we)
    resp = np.nanmean(avg_epochs[:, :, mask], axis=2)   # (288, n_ch)
    # NaN-fill: column mean; all-NaN column → 0
    col_means = np.nanmean(resp, axis=0, keepdims=True)
    col_means = np.where(np.isnan(col_means), 0.0, col_means)
    return np.where(np.isnan(resp), col_means, resp)

print('Running RSA...')
t0 = time.perf_counter()

for wi, (ws, we) in enumerate(windows):
    resp_p13 = _window_resp(avg_p13, t_ms, ws, we)
    resp_p14 = _window_resp(avg_p14, t_ms, ws, we)

    rdm_p13 = pdist(resp_p13, metric='correlation')
    rdm_p14 = pdist(resp_p14, metric='correlation')

    ecog_rdm = (rdm_p13 + rdm_p14) / 2.0   # average per-subject RDMs

    # Inter-subject reliability
    reliability[wi] = spearmanr(rdm_p13, rdm_p14).statistic

    for li, (arch, lk) in enumerate(cnn_layer_order):
        if not np.all(np.isfinite(ecog_rdm)):
            continue
        rsa_matrix[wi, li] = spearmanr(ecog_rdm, cnn_rdms[(arch, lk)]).statistic

print(f'Done in {time.perf_counter() - t0:.1f}s')
print(f'RSA range: [{np.nanmin(rsa_matrix):.3f}, {np.nanmax(rsa_matrix):.3f}]')
print(f'Inter-subject reliability range: [{np.nanmin(reliability):.3f}, {np.nanmax(reliability):.3f}]')

# Save
pd.DataFrame(
    rsa_matrix,
    index=[f'{int(w[0])}_{int(w[1])}ms' for w in windows],
    columns=[f'{a}|{lk}' for a, lk in cnn_layer_order],
).to_csv(RESULTS / 'milestone_04c_rsa_matrix.csv')


44 windows × 42 CNN layers
Running RSA...


In [ ]:
# Compute arch boundary positions and short x-tick labels
arch_boundaries = []
xtick_labels = []
prev_arch = None
for li, (arch, lk) in enumerate(cnn_layer_order):
    if arch != prev_arch:
        arch_boundaries.append(li)
        prev_arch = arch
    short = lk.split('.')[-1].replace('@step_', 'r')
    xtick_labels.append(short)

arch_boundaries_ext = arch_boundaries + [n_cnn]

fig, (ax_heat, ax_rel) = plt.subplots(
    2, 1, figsize=(15, 8),
    gridspec_kw={'height_ratios': [5, 1], 'hspace': 0.08},
    sharex=True,
)

# Heatmap
im = ax_heat.imshow(
    rsa_matrix, aspect='auto', origin='upper', cmap='RdBu_r',
    vmin=-0.05, vmax=0.35,
    extent=[-0.5, n_cnn - 0.5, n_win - 0.5, -0.5],
)
plt.colorbar(im, ax=ax_heat, label="Spearman r", shrink=0.8)

# Arch boundaries
for xb in arch_boundaries[1:]:
    ax_heat.axvline(xb - 0.5, color='white', lw=1.5, ls='--', alpha=0.8)

# Arch labels
for i in range(len(arch_boundaries)):
    mid = (arch_boundaries_ext[i] + arch_boundaries_ext[i + 1]) / 2 - 0.5
    ax_heat.text(mid, -1.2, ALL_ARCHS[i] if i < len(ALL_ARCHS) else '',
                 ha='center', va='bottom', fontsize=8, fontweight='bold',
                 transform=ax_heat.get_xaxis_transform())

# Horizontal time markers
ytick_vals  = np.arange(0, n_win, 4)
ytick_labels_ms = [f'{win_centers_ms[i]:.0f}' for i in ytick_vals]
for t_mark, label in [(0, 'onset'), (50, '50ms'), (200, '200ms')]:
    wi_mark = int(np.argmin(np.abs(win_centers_ms - t_mark)))
    ax_heat.axhline(wi_mark, color='black', lw=1.2, ls='--', alpha=0.7, label=label)

ax_heat.set_yticks(ytick_vals)
ax_heat.set_yticklabels(ytick_labels_ms, fontsize=8)
ax_heat.set_ylabel('Window centre (ms)')
ax_heat.set_title('RSA: ECoG × CNN RDM (Spearman r) — 25ms sliding windows, per-subject RDMs averaged',
                  fontsize=10)
ax_heat.legend(loc='upper right', fontsize=8, framealpha=0.7)

# Reliability curve
ax_rel.plot(np.arange(n_win), reliability, color='navy', lw=1.5)
ax_rel.axhline(0, color='gray', lw=0.5)
ax_rel.fill_between(np.arange(n_win), 0, reliability,
                    where=reliability > 0, alpha=0.25, color='navy')
ax_rel.set_ylabel('Inter-sub r', fontsize=8)
ax_rel.set_yticks([0, 0.2, 0.4])
ax_rel.set_xticks(range(n_cnn))
ax_rel.set_xticklabels(xtick_labels, rotation=70, fontsize=6)
ax_rel.set_xlabel('CNN layer (grouped by architecture)')
for xb in arch_boundaries[1:]:
    ax_rel.axvline(xb - 0.5, color='gray', lw=1.0, ls='--', alpha=0.5)

plt.tight_layout()
out = RESULTS / 'milestone_04c_rsa_heatmap.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')


## Analysis 2 — Geodesic Separability via Isomap

Fisher ratio assumes a Euclidean flat space. Visual representations in high-dimensional
CNN layers may lie on a curved manifold, so Euclidean distances underestimate true
category separation.

Isomap approximates geodesic distances by building a k-NN graph, then uses MDS on
the resulting geodesic distance matrix. Fisher ratio computed in the 10D Isomap
embedding measures separability in the manifold geometry.

In [ ]:
def fisher_ratio(X, labels):
    """Between-class / within-class scatter. Higher = better separability."""
    classes = np.unique(labels)
    mu      = X.mean(axis=0)
    between = sum(
        np.sum(labels == c) * float(np.sum((X[labels == c].mean(0) - mu) ** 2))
        for c in classes
    )
    within  = sum(
        float(np.sum((X[labels == c] - X[labels == c].mean(0)) ** 2))
        for c in classes
    )
    return between / within if within > 0 else 0.0

sep_euclidean = pd.read_csv(RESULTS / 'milestone_04_separability.csv')
print('Euclidean separability loaded:')
print(sep_euclidean.pivot_table(index='layer_depth', columns='arch', values='fisher_full').round(3))


In [ ]:
N_NEIGHBORS  = 10
N_COMPONENTS = 10

isomap_rows = []
print(f'Isomap: n_neighbors={N_NEIGHBORS}, n_components={N_COMPONENTS}')

for arch in ALL_ARCHS:
    p = RESULTS / f'cnn_features_{arch}.pkl'
    if not p.exists():
        print(f'[skip] {arch}')
        continue
    cache = load_feature_cache(p)
    cats  = cache.metadata['category'].values

    for depth, lk in enumerate(cache.features.keys()):
        X_std = StandardScaler().fit_transform(cache.features[lk].astype('float32'))
        try:
            X_iso = Isomap(n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS,
                           n_jobs=-1).fit_transform(X_std)
            fr_geo = fisher_ratio(X_iso, cats)
        except Exception as e:
            print(f'  [isomap err] {arch}/{lk}: {e}')
            fr_geo = np.nan

        isomap_rows.append({'arch': arch, 'layer': lk,
                            'layer_depth': depth, 'fisher_geodesic': fr_geo})
        print(f'  {arch:16s} [{depth}] {lk:32s}  geodesic={fr_geo:.3f}')

iso_df = pd.DataFrame(isomap_rows)
iso_df.to_csv(RESULTS / 'milestone_04c_isomap_fisher.csv', index=False)
print(f'\nSaved → {RESULTS / "milestone_04c_isomap_fisher.csv"}')


In [ ]:
merged = sep_euclidean.merge(
    iso_df[['arch', 'layer', 'layer_depth', 'fisher_geodesic']],
    on=['arch', 'layer', 'layer_depth'], how='outer',
)
merged['delta'] = merged['fisher_geodesic'] - merged['fisher_full']

n_archs = len(ALL_ARCHS)
ncols = 4
nrows = (n_archs + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5), sharey=False)
axes_flat = axes.flatten()

for ai, arch in enumerate(ALL_ARCHS):
    ax  = axes_flat[ai]
    sub = merged[merged['arch'] == arch].sort_values('layer_depth').dropna(subset=['layer_depth'])
    ax.plot(sub['layer_depth'], sub['fisher_full'],
            color='steelblue', marker='o', markersize=5, lw=2, label='Euclidean')
    ax.plot(sub['layer_depth'], sub['fisher_geodesic'],
            color='tomato', marker='s', markersize=5, lw=2, ls='--', label='Geodesic')
    ax.set_title(arch, fontsize=9, fontweight='bold')
    ax.set_xlabel('Layer depth', fontsize=8)
    ax.set_ylabel('Fisher ratio', fontsize=8)
    ax.set_xticks(sub['layer_depth'].values)
    ax.set_xticklabels(
        [lk.split('.')[-1][:9] for lk in sub['layer'].values],
        rotation=45, fontsize=6,
    )
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

for ai in range(n_archs, len(axes_flat)):
    axes_flat[ai].set_visible(False)

fig.suptitle(f'Euclidean vs Geodesic (Isomap {N_NEIGHBORS}NN, {N_COMPONENTS}D) Fisher Ratio',
             fontsize=11)
plt.tight_layout()
out = RESULTS / 'milestone_04c_geodesic_vs_euclidean.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for arch in ALL_ARCHS:
    sub = merged[merged['arch'] == arch].sort_values('layer_depth').dropna(subset=['delta'])
    if sub.empty: continue
    ls = '--' if arch == 'cornet_s' else '-'
    ax.plot(sub['layer_depth'], sub['delta'], marker='o', markersize=4,
            lw=1.8, ls=ls, label=arch)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Layer depth')
ax.set_ylabel('Δ Fisher (Geodesic − Euclidean)')
ax.set_title('Non-linearity benefit per layer\n'
             'Positive = manifold embedding reveals more category structure', fontsize=10)
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
out = RESULTS / 'milestone_04c_delta_fisher.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

summary = (merged.dropna(subset=['delta'])
           .sort_values('delta', ascending=False)
           .drop_duplicates('arch')
           [['arch', 'layer', 'layer_depth', 'fisher_full', 'fisher_geodesic', 'delta']]
           .reset_index(drop=True))
summary.to_csv(RESULTS / 'milestone_04c_delta_fisher_summary.csv', index=False)
print('Top layer by geodesic benefit per arch:')
print(summary.to_string(index=False))
print(f'Saved → {RESULTS / "milestone_04c_delta_fisher_summary.csv"}')


## Analysis 3 — Scrambled Check: Late-Onset Gamma Specificity

Scrambled stimuli share the low-level statistics of natural images but lack
high-level structure. If the ECoG broadband response reflects genuine feature
encoding rather than low-level drive, scrambled stimuli should show:

- **Early [50–100ms]:** similar response to natural (both driven by low-level features)
- **Mid [100–200ms]:** partial separation
- **Late [200–500ms]:** significantly *lower* response than natural categories

Permutation tests (n=1000) on image-level responses confirm window-by-window specificity.

In [ ]:
TIME_WINDOWS = {'early': (50, 100), 'mid': (100, 200), 'late': (200, 500)}
WIN_COLORS   = {'early': '#ffffb3', 'mid': '#b3e2cd', 'late': '#fdcdac'}

# Use p13 + p14 concatenated for time courses (qualitative, not quantitative)
# Mean across visual channels per subject, then average across subjects
def subject_mean_traces(avg_epochs):
    """(288, n_ch, n_t) → (288, n_t) mean across channels."""
    return np.nanmean(avg_epochs, axis=1)

traces_p13 = subject_mean_traces(avg_p13)   # (288, n_t)
traces_p14 = subject_mean_traces(avg_p14)

# Average subjects (equal weight; both have same 288 images)
mean_traces = np.nanmean(np.stack([traces_p13, traces_p14], axis=0), axis=0)  # (288, n_t)

fig, ax = plt.subplots(figsize=(12, 5))
for wname, (ws, we) in TIME_WINDOWS.items():
    ax.axvspan(ws, we, alpha=0.18, color=WIN_COLORS[wname], label=f'{wname} [{ws}–{we}ms]')

for cat in CATEGORY_NAMES:
    cat_ids    = bank.metadata.index[bank.metadata['category'] == cat].values
    cat_traces = mean_traces[cat_ids]                    # (48, n_t)
    cat_mean   = np.nanmean(cat_traces, axis=0)
    cat_sem    = np.nanstd(cat_traces, axis=0) / np.sqrt(cat_traces.shape[0])
    ax.plot(t_ms, cat_mean, color=CATEGORY_COLORS[cat], lw=2, label=cat)
    ax.fill_between(t_ms, cat_mean - cat_sem, cat_mean + cat_sem,
                    color=CATEGORY_COLORS[cat], alpha=0.15)

ax.axvline(0, color='black', lw=1.5, ls='--')
ax.axhline(0, color='gray', lw=0.5)
ax.set_xlim(-100, 1000)
ax.set_xlabel('Time from stimulus onset (ms)', fontsize=11)
ax.set_ylabel('Broadband gamma (fractional change)', fontsize=11)
ax.set_title('Trial-averaged broadband gamma by category\n'
             '(mean ± SEM across 48 images, averaged over visual electrodes & subjects)', fontsize=10)
ax.legend(fontsize=9, ncol=4, loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
out = RESULTS / 'milestone_04c_timecourse_by_category.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')


In [ ]:
N_PERMS = 1000
RNG     = np.random.default_rng(42)

def window_scalar(traces, t_ms, ws, we):
    """Mean response in [ws, we) ms per image. Returns (n_images,)."""
    mask = (t_ms >= ws) & (t_ms < we)
    return np.nanmean(traces[:, mask], axis=1)

def permutation_p(a, b, n_perms=1000, rng=None):
    """Two-sided permutation test: H0 = no difference in means."""
    rng = rng or np.random.default_rng(0)
    obs  = np.nanmean(a) - np.nanmean(b)
    pool = np.concatenate([a, b])
    n_a  = len(a)
    null = np.array([rng.permutation(pool)[:n_a].mean() -
                     rng.permutation(pool)[n_a:].mean()
                     for _ in range(n_perms)])
    return obs, float(np.mean(np.abs(null) >= np.abs(obs)))

scr_ids    = bank.metadata.index[bank.metadata['category'] == 'scrambled'].values
nat_cats   = [c for c in CATEGORY_NAMES if c != 'scrambled']
scr_traces = mean_traces[scr_ids]

perm_rows = []
print(f'{"Window":<20} {"Category":<12} {"Obs diff":>9} {"p-value":>9} {"Sig":>5}')
print('-' * 60)

for wname, (ws, we) in TIME_WINDOWS.items():
    scr_resp = window_scalar(scr_traces, t_ms, ws, we)
    for cat in nat_cats:
        nat_ids  = bank.metadata.index[bank.metadata['category'] == cat].values
        nat_resp = window_scalar(mean_traces[nat_ids], t_ms, ws, we)
        obs, p   = permutation_p(nat_resp, scr_resp, N_PERMS, RNG)
        sig      = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        perm_rows.append({'window': wname, 'window_ms': f'{ws}-{we}', 'category': cat,
                          'obs_diff': obs, 'p_value': p, 'sig': sig})
        print(f'{wname} [{ws}-{we}ms]{"":<5} {cat:<12} {obs:>9.4f} {p:>9.4f} {sig:>5}')

perm_df = pd.DataFrame(perm_rows)
perm_df.to_csv(RESULTS / 'milestone_04c_permutation_tests.csv', index=False)
print(f'\nSaved → {RESULTS / "milestone_04c_permutation_tests.csv"}')


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x     = np.arange(len(nat_cats))
width = 0.25

for wi, (wname, (ws, we)) in enumerate(TIME_WINDOWS.items()):
    w_data    = perm_df[perm_df['window'] == wname]
    obs_diffs = [w_data[w_data['category'] == c]['obs_diff'].values[0] for c in nat_cats]
    p_vals    = [w_data[w_data['category'] == c]['p_value'].values[0]  for c in nat_cats]
    bars = ax.bar(x + wi * width, obs_diffs, width=width,
                  color=WIN_COLORS[wname], edgecolor='black', lw=0.7,
                  label=f'{wname} [{ws}–{we}ms]')
    for xi, (od, pv) in enumerate(zip(obs_diffs, p_vals)):
        if pv < 0.05:
            mk = '***' if pv < 0.001 else ('**' if pv < 0.01 else '*')
            ax.text(xi + wi * width, max(0, od) + 0.002, mk,
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels([c.capitalize() for c in nat_cats], fontsize=10)
ax.set_ylabel('Natural − Scrambled mean gamma', fontsize=10)
ax.set_title('Scrambled check — natural vs scrambled broadband gamma per time window\n'
             '(* p<0.05, ** p<0.01, *** p<0.001, permutation test n=1000)', fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.2, axis='y')
plt.tight_layout()
out = RESULTS / 'milestone_04c_scrambled_check.png'
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')


In [ ]:
print('=' * 65)
print('MILESTONE 04c — SUMMARY')
print('=' * 65)

# RSA peak
peak_coords = np.unravel_index(np.nanargmax(rsa_matrix), rsa_matrix.shape)
pw, pl = peak_coords
print(f'\nAnalysis 1 — RSA')
print(f'  Peak r = {rsa_matrix[pw, pl]:.3f}')
print(f'  Window : {windows[pw, 0]}–{windows[pw, 1]} ms')
arch_pk, lk_pk = cnn_layer_order[pl]
print(f'  Layer  : {arch_pk} / {lk_pk}')
print(f'  Inter-subject reliability at peak window: {reliability[pw]:.3f}')

print(f'\nAnalysis 2 — Geodesic Fisher (Isomap {N_NEIGHBORS}NN, {N_COMPONENTS}D)')
print('  Top 3 layers by geodesic benefit:')
top3 = (merged.dropna(subset=['delta'])
        .nlargest(3, 'delta')[['arch','layer','layer_depth','fisher_full','fisher_geodesic','delta']])
print(top3.to_string(index=False))

print(f'\nAnalysis 3 — Scrambled check')
piv = perm_df.pivot_table(index='window', columns='category', values='sig', aggfunc='first')
print(piv[nat_cats].to_string())

print('\nOutput files:')
for fn in sorted(RESULTS.glob('milestone_04c_*')):
    print(f'  {fn.name:<55}  {fn.stat().st_size / 1e3:.0f} KB')
